In [ ]:
############################################################
## Script to quantify the effect of in silico perturbation
## of the lincRNA on the expression of TUSC3
##
## Author: Shelley
## Last Date: 2026-02-28
############################################################

In [ ]:
# step.1 setup and imports
from alphagenome import colab_utils
from alphagenome.data import gene_annotation, genome, track_data, transcript
from alphagenome.models import dna_client
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

dna_model = dna_client.create(api_key="YOUR_API_KEY")

# Load metadata objects for human.
output_metadata = dna_model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
)

# Load gene annotations (from GENCODE).
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

E0228 09:12:40.531667 61703193 ssl_transport_security_utils.cc:114] Corruption detected.
E0228 09:12:40.531708 61703193 ssl_transport_security_utils.cc:71] error:1e000065:Cipher functions:OPENSSL_internal:BAD_DECRYPT
E0228 09:12:40.531716 61703193 ssl_transport_security_utils.cc:71] error:1000008b:SSL routines:OPENSSL_internal:DECRYPTION_FAILED_OR_BAD_RECORD_MAC
E0228 09:12:40.531719 61703193 secure_endpoint.cc:254] Decryption error: TSI_DATA_CORRUPTED


In [3]:
############################################################
# Step 2: Load prediction outputs
#   - Reference prediction output (out_ref)
############################################################

import os
import pickle

REF_PKL = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/alphagenome_out_ref.pkl"

if not os.path.exists(REF_PKL):
    raise FileNotFoundError(f"Reference prediction file not found: {REF_PKL}")

with open(REF_PKL, "rb") as f:
    out_ref = pickle.load(f)

print("Loaded reference prediction:")
print("  path:", REF_PKL)
print("  type:", type(out_ref))
print("  has rna_seq:", hasattr(out_ref, "rna_seq"))
print("  has cage:", hasattr(out_ref, "cage"))

Loaded reference prediction:
  path: /Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/alphagenome_out_ref.pkl
  type: <class 'alphagenome.models.dna_output.Output'>
  has rna_seq: True
  has cage: True


In [4]:
############################################################
# Step 3: Define prediction regions (hg38)
############################################################

# Input coordinates (as typically reported: 1-based, inclusive)
lincRNA_chr = "chr8"
lincRNA_start_1based = 15_973_315
lincRNA_end_1based   = 16_000_324

tusc3_chr = "chr8"
tusc3_start_1based = 15_417_188
tusc3_end_1based   = 15_852_091

# Convert to AlphaGenome Interval coordinates (0-based start, end-exclusive)
lincRNA_interval = genome.Interval(
    chromosome=lincRNA_chr,
    start=lincRNA_start_1based - 1,
    end=lincRNA_end_1based,  # inclusive -> exclusive
    strand="+",
    name="ENSG00000253648"
)

tusc3_interval = genome.Interval(
    chromosome=tusc3_chr,
    start=tusc3_start_1based - 1,
    end=tusc3_end_1based,
    strand="+",
    name="TUSC3"
)

# Span that covers both loci (minimal joint span)
joint_span = genome.Interval(
    chromosome="chr8",
    start=min(lincRNA_interval.start, tusc3_interval.start),
    end=max(lincRNA_interval.end, tusc3_interval.end),
    strand=".",
    name="ENSG00000253648_to_TUSC3_span"
)

# Resize to AlphaGenome required input length (1 Mb)
interval = joint_span.resize(dna_client.SEQUENCE_LENGTH_1MB)

joint_span, interval
# sanity check: print the width for these regions
print("lincRNA_interval width (bp):", lincRNA_interval.width)
print("tusc3_interval width (bp):", tusc3_interval.width)
print("joint_span width (bp):", joint_span.width)
print("interval (1Mb) width (bp):", interval.width)

lincRNA_interval width (bp): 27010
tusc3_interval width (bp): 434904
joint_span width (bp): 583137
interval (1Mb) width (bp): 1048576


In [7]:
############################################################
# Step 3A: Extract transcripts overlapping the 1Mb interval
# (no manual filtering)
############################################################

from alphagenome.data import transcript as ag_transcript

# Build extractor directly from the full GTF feather dataframe
tx_extractor = ag_transcript.TranscriptExtractor(gtf)

# Extract all transcripts overlapping the 1Mb interval
tx_all = tx_extractor.extract(interval)

print("# transcripts overlapping 1Mb interval:", len(tx_all))

# quick peek
for t in tx_all[:20]:
    print({
        "gene_id": getattr(t, "gene_id", None),
        "transcript_id": getattr(t, "transcript_id", None),
        "strand": getattr(t, "strand", None),
        "chromosome": getattr(t, "chromosome", None),
        "start": getattr(t, "start", None),
        "end": getattr(t, "end", None),
    })

# transcripts overlapping 1Mb interval: 29
{'gene_id': 'ENSG00000038945.15', 'transcript_id': 'ENST00000262101.10', 'strand': '-', 'chromosome': 'chr8', 'start': None, 'end': None}
{'gene_id': 'ENSG00000038945.15', 'transcript_id': 'ENST00000350896.3', 'strand': '-', 'chromosome': 'chr8', 'start': None, 'end': None}
{'gene_id': 'ENSG00000038945.15', 'transcript_id': 'ENST00000355282.6', 'strand': '-', 'chromosome': 'chr8', 'start': None, 'end': None}
{'gene_id': 'ENSG00000038945.15', 'transcript_id': 'ENST00000381998.8', 'strand': '-', 'chromosome': 'chr8', 'start': None, 'end': None}
{'gene_id': 'ENSG00000104723.21', 'transcript_id': 'ENST00000382020.8', 'strand': '+', 'chromosome': 'chr8', 'start': None, 'end': None}
{'gene_id': 'ENSG00000185053.14', 'transcript_id': 'ENST00000382080.6', 'strand': '-', 'chromosome': 'chr8', 'start': None, 'end': None}
{'gene_id': 'ENSG00000038945.15', 'transcript_id': 'ENST00000445506.6', 'strand': '-', 'chromosome': 'chr8', 'start': None, 'end': Non

In [8]:
############################################################
# Step 3B: Subset tx_all to TUSC3 + lncRNA and summarize
############################################################

import pandas as pd

TUSC3_GENE_ID = "ENSG00000104723"
LINC_GENE_ID  = "ENSG00000253648"

def strip_version(x):
    return x.split(".")[0] if isinstance(x, str) and x else None

def summarize_transcript_list(txs, label):
    rows = []
    for t in txs:
        tx_iv = getattr(t, "transcript_interval", None)
        info = getattr(t, "info", {}) or {}

        rows.append({
            "label": label,
            "gene_id": getattr(t, "gene_id", None),
            "gene_id_base": strip_version(getattr(t, "gene_id", None)),
            "transcript_id": getattr(t, "transcript_id", None),
            "chromosome": getattr(t, "chromosome", None),
            "strand": getattr(t, "strand", None),
            "strand_int": getattr(t, "strand_int", None),
            "is_coding": getattr(t, "is_coding", None),

            # transcript span
            "tx_start": getattr(tx_iv, "start", None) if tx_iv is not None else None,
            "tx_end":   getattr(tx_iv, "end", None)   if tx_iv is not None else None,
            "tx_width": getattr(tx_iv, "width", None) if tx_iv is not None else None,

            # feature counts (if present)
            "n_exons":   len(getattr(t, "exons", []) or []),
            "n_introns": len(getattr(t, "introns", []) or []),
            "n_cds":     len(getattr(t, "cds", []) or []),
            "n_utr5":    len(getattr(t, "utr5", []) or []),
            "n_utr3":    len(getattr(t, "utr3", []) or []),

            # common annotation keys if present in info
            "gene_name": info.get("gene_name", None),
            "transcript_type": info.get("transcript_type", None),
        })

    df = pd.DataFrame(rows).sort_values(["tx_width", "transcript_id"], ascending=[False, True])
    print(f"\n{label}: {df.shape[0]} transcript(s)")
    display(df)
    return df

# --- subset tx_all ---
tx_tusc3 = [t for t in tx_all if strip_version(getattr(t, "gene_id", None)) == TUSC3_GENE_ID]
tx_linc  = [t for t in tx_all if strip_version(getattr(t, "gene_id", None)) == LINC_GENE_ID]

# --- summarize ---
df_tusc3 = summarize_transcript_list(tx_tusc3, "TUSC3")
df_linc  = summarize_transcript_list(tx_linc,  "lncRNA_ENSG00000253648")


TUSC3: 13 transcript(s)


,label,gene_id,gene_id_base,transcript_id,chromosome,strand,strand_int,is_coding,tx_start,tx_end,tx_width,n_exons,n_introns,n_cds,n_utr5,n_utr3,gene_name,transcript_type
1,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000503191.5,chr8,+,1,False,15417214,15673836,256622,6,5,0,0,0,TUSC3,protein_coding_CDS_not_defined
0,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000382020.8,chr8,+,1,True,15540222,15766649,226427,10,9,10,1,1,TUSC3,protein_coding
2,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000503731.6,chr8,+,1,True,15540262,15766638,226376,11,10,10,1,2,TUSC3,protein_coding
12,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000515859.5,chr8,+,1,True,15540226,15764661,224435,9,8,7,1,3,TUSC3,nonsense_mediated_decay
3,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000506802.5,chr8,+,1,True,15540232,15764613,224381,9,8,9,1,1,TUSC3,protein_coding
9,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000510836.5,chr8,+,1,True,15540278,15764629,224351,8,7,7,1,2,TUSC3,protein_coding
8,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000509380.5,chr8,+,1,True,15540280,15733645,193365,8,7,8,1,1,TUSC3,protein_coding
11,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000511783.2,chr8,+,1,True,15623079,15758277,135198,10,9,9,0,2,TUSC3,protein_coding
5,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000507400.1,chr8,+,1,False,15561618,15673836,112218,6,5,0,0,0,TUSC3,protein_coding_CDS_not_defined
6,TUSC3,ENSG00000104723.21,ENSG00000104723,ENST00000508446.1,chr8,+,1,False,15673762,15764418,90656,3,2,0,0,0,TUSC3,protein_coding_CDS_not_defined



lncRNA_ENSG00000253648: 1 transcript(s)


,label,gene_id,gene_id_base,transcript_id,chromosome,strand,strand_int,is_coding,tx_start,tx_end,tx_width,n_exons,n_introns,n_cds,n_utr5,n_utr3,gene_name,transcript_type
0,lncRNA_ENSG00000253648,ENSG00000253648.1,ENSG00000253648,ENST00000517369.1,chr8,+,1,False,15973314,16000324,27010,3,2,0,0,0,ENSG00000253648,lncRNA


In [9]:
############################################################
# Step 3C: For TUSC3, retrieve the transcript with the highest evidence
# Strategy:
#   1) keep protein-coding transcripts
#   2) keep TSL == 1 (highest transcript support level)
#   3) among remaining TUSC3 transcripts, pick the longest transcript
############################################################

from alphagenome.data import gene_annotation
from alphagenome.data import transcript as ag_transcript

TUSC3_GENE_ID = "ENSG00000104723"

def strip_version(x):
    return x.split(".")[0] if isinstance(x, str) and x else None

# 1) Filter to protein-coding + TSL1
gtf_pc = gene_annotation.filter_protein_coding(gtf)
gtf_pc_tsl1 = gene_annotation.filter_transcript_support_level(gtf_pc, ["1"])

# 2) Build TranscriptExtractor on the filtered GTF and extract within your 1Mb interval
tx_extractor_tsl1 = ag_transcript.TranscriptExtractor(gtf_pc_tsl1)
tx_all_tsl1 = tx_extractor_tsl1.extract(interval)

print("# TSL1 protein-coding transcripts in 1Mb interval:", len(tx_all_tsl1))


# TSL1 protein-coding transcripts in 1Mb interval: 6


In [10]:
# 3) Subset to TUSC3 transcripts (gene_id in GTF is versioned, e.g. ENSG... .21)
tx_tusc3_tsl1 = [
    t for t in tx_all_tsl1
    if strip_version(getattr(t, "gene_id", None)) == TUSC3_GENE_ID
]

print("# TUSC3 TSL1 protein-coding transcripts:", len(tx_tusc3_tsl1))

if len(tx_tusc3_tsl1) == 0:
    raise ValueError("No TUSC3 transcripts found after protein-coding + TSL1 filtering.")

# 4) Pick “highest evidence” transcript as the longest among TSL1 set
def tx_width(t):
    tx_iv = getattr(t, "transcript_interval", None)
    return getattr(tx_iv, "width", -1) if tx_iv is not None else -1

tusc3_best_tx = max(tx_tusc3_tsl1, key=tx_width)

print("\nSelected TUSC3 transcript (highest evidence):")
print("  gene_id:", getattr(tusc3_best_tx, "gene_id", None))
print("  transcript_id:", getattr(tusc3_best_tx, "transcript_id", None))
print("  strand:", getattr(tusc3_best_tx, "strand", None))
print("  transcript_interval:", getattr(tusc3_best_tx, "transcript_interval", None))
print("  n_exons:", len(getattr(tusc3_best_tx, "exons", []) or []))

# TUSC3 TSL1 protein-coding transcripts: 2

Selected TUSC3 transcript (highest evidence):
  gene_id: ENSG00000104723.21
  transcript_id: ENST00000382020.8
  strand: +
  transcript_interval: chr8:15540222-15766649:+
  n_exons: 10


In [11]:
############################################################
# Step 3D (Option A): Combine 1 best transcript for TUSC3 and lncRNA
#   - TUSC3: tusc3_best_tx (from Step 3C)
#   - lncRNA: choose the longest transcript in tx_all for ENSG00000253648
############################################################

LINC_GENE_ID = "ENSG00000253648"

def strip_version(x):
    return x.split(".")[0] if isinstance(x, str) and x else None

# lncRNA transcripts from tx_all (already extracted in the 1Mb interval)
tx_linc = [t for t in tx_all if strip_version(getattr(t, "gene_id", None)) == LINC_GENE_ID]
print("# lncRNA transcripts in interval:", len(tx_linc))

if len(tx_linc) == 0:
    raise ValueError("No lncRNA transcripts found in tx_all for ENSG00000253648.")

def tx_width(t):
    tx_iv = getattr(t, "transcript_interval", None)
    return getattr(tx_iv, "width", -1) if tx_iv is not None else -1

linc_best_tx = max(tx_linc, key=tx_width)

print("Selected lncRNA transcript:")
print("  gene_id:", getattr(linc_best_tx, "gene_id", None))
print("  transcript_id:", getattr(linc_best_tx, "transcript_id", None))
print("  transcript_interval:", getattr(linc_best_tx, "transcript_interval", None))

# Combine into one list for downstream plotting/annotation
combined_transcripts = [tusc3_best_tx, linc_best_tx]

print("\nCombined transcripts:")
for t in combined_transcripts:
    print({
        "gene_id": getattr(t, "gene_id", None),
        "transcript_id": getattr(t, "transcript_id", None),
        "strand": getattr(t, "strand", None),
        "span": str(getattr(t, "transcript_interval", None)),
    })

# lncRNA transcripts in interval: 1
Selected lncRNA transcript:
  gene_id: ENSG00000253648.1
  transcript_id: ENST00000517369.1
  transcript_interval: chr8:15973314-16000324:+

Combined transcripts:
{'gene_id': 'ENSG00000104723.21', 'transcript_id': 'ENST00000382020.8', 'strand': '+', 'span': 'chr8:15540222-15766649:+'}
{'gene_id': 'ENSG00000253648.1', 'transcript_id': 'ENST00000517369.1', 'strand': '+', 'span': 'chr8:15973314-16000324:+'}


In [18]:
############################################################
# Step 4A (updated): RNA-seq ROI = union of TUSC3 exons
#   - Use highest-evidence TUSC3 transcript
#   - Build boolean mask over the 1Mb interval
############################################################

import numpy as np

# Ensure transcript exists
if tusc3_best_tx is None:
    raise ValueError("tusc3_best_tx is not defined.")

# Get exon intervals
tusc3_exons = getattr(tusc3_best_tx, "exons", None)
if tusc3_exons is None or len(tusc3_exons) == 0:
    raise ValueError("No exon intervals found for tusc3_best_tx.")

print("Number of exons:", len(tusc3_exons))

# Create a mask across the full 1Mb prediction interval
rna_roi_mask = np.zeros(interval.width, dtype=bool)

for ex in tusc3_exons:
    if not interval.contains(ex):
        raise ValueError(f"Exon not fully inside prediction interval: {ex}")
    
    # Convert exon genomic coordinates to relative positions within 1Mb window
    rel_start = ex.start - interval.start
    rel_end   = ex.end   - interval.start
    
    rna_roi_mask[rel_start:rel_end] = True

print("Total RNA-seq ROI length (bp):", rna_roi_mask.sum())
print("Expected exon sum length:",
      sum(ex.width for ex in tusc3_exons))

############################################################
# Print TUSC3 exon genomic coordinates (manual review)
############################################################

print("\nTUSC3 exons (genomic coordinates):\n")

for i, ex in enumerate(tusc3_exons, start=1):
    print(f"Exon {i:02d}: {ex.chromosome}:{ex.start}-{ex.end} ({ex.strand}) | width={ex.width}")

Number of exons: 10
Total RNA-seq ROI length (bp): 3683
Expected exon sum length: 3683

TUSC3 exons (genomic coordinates):

Exon 01: chr8:15540222-15540568 (+) | width=346
Exon 02: chr8:15623079-15623249 (+) | width=170
Exon 03: chr8:15650696-15650814 (+) | width=118
Exon 04: chr8:15659506-15659647 (+) | width=141
Exon 05: chr8:15662155-15662296 (+) | width=141
Exon 06: chr8:15673746-15673836 (+) | width=90
Exon 07: chr8:15730665-15730729 (+) | width=64
Exon 08: chr8:15743537-15743612 (+) | width=75
Exon 09: chr8:15748374-15748465 (+) | width=91
Exon 10: chr8:15764202-15766649 (+) | width=2447


In [19]:
############################################################
# Step 4B: CAGE ROI = 501-bp window centered at TUSC3 TSS
#   - 250 bp upstream of TSS
#   - TSS base
#   - 250 bp downstream of TSS
# Coordinate system: 0-based start, end-exclusive
############################################################

from alphagenome.data import genome

# transcript interval for the selected highest-evidence TUSC3 transcript
tx_iv = getattr(tusc3_best_tx, "transcript_interval", None)
if tx_iv is None:
    raise ValueError("tusc3_best_tx has no transcript_interval.")

strand = getattr(tusc3_best_tx, "strand", None)
if strand not in {"+", "-"}:
    raise ValueError(f"Unexpected strand for tusc3_best_tx: {strand}")

# TSS in 0-based coordinates
tss0 = tx_iv.start if strand == "+" else (tx_iv.end - 1)

# 501-bp centered window: [tss-250, tss+251)
cage_roi = genome.Interval(
    chromosome=tx_iv.chromosome,
    start=tss0 - 250,
    end=tss0 + 251,
    strand=strand,
    name="TUSC3_TSS_501bp"
)

# sanity checks
assert cage_roi.width == 501, cage_roi.width
if not interval.contains(cage_roi):
    raise ValueError(f"CAGE ROI not fully contained in 1Mb interval:\n  cage_roi={cage_roi}\n  interval={interval}")

print("TUSC3 transcript:", getattr(tusc3_best_tx, "transcript_id", None))
print("TSS (0-based):", tss0, "| strand:", strand)
print("CAGE ROI:", cage_roi)
print("CAGE ROI width (bp):", cage_roi.width)

# optional: also print relative coordinates within the 1Mb interval
rel_start = cage_roi.start - interval.start
rel_end   = cage_roi.end   - interval.start
print("CAGE ROI relative to 1Mb interval:", f"{rel_start}-{rel_end}")

TUSC3 transcript: ENST00000382020.8
TSS (0-based): 15540222 | strand: +
CAGE ROI: chr8:15539972-15540473:+
CAGE ROI width (bp): 501
CAGE ROI relative to 1Mb interval: 355504-356005


In [20]:
############################################################
# Step 5A (REF): Summarize RNA-seq coverage in TUSC3 exons
#   Outputs:
#     1) per_exon_rna_summary         (long: exon × track)
#     2) gene_rna_summary             (per-track over union of exons)
#     3) gene_rna_summary_collapsed   (collapsed across tracks)
############################################################

import numpy as np
import pandas as pd

# --- inputs assumed ---
# out_ref        : AlphaGenome Output (reference prediction)
# interval       : 1Mb genome.Interval
# tusc3_exons    : list[genome.Interval] (exons of tusc3_best_tx)
# rna_roi_mask   : np.ndarray[bool] length == interval.width (union of exons)

ref_rna_t = out_ref.rna_seq
Vref_rna = np.asarray(ref_rna_t.values)  # (pos, tracks)
meta_rna = ref_rna_t.metadata.copy().reset_index(drop=True)

# --- sanity checks ---
assert Vref_rna.shape[0] == interval.width, (Vref_rna.shape[0], interval.width)
assert rna_roi_mask.shape[0] == interval.width
assert rna_roi_mask.dtype == bool
assert rna_roi_mask.sum() > 0, "Exon ROI mask is empty."
assert len(tusc3_exons) > 0, "No exons found for TUSC3 transcript."

print("RNA-seq values shape (pos, tracks):", Vref_rna.shape)
print("Number of exons:", len(tusc3_exons))
print("Total exonic bp (union):", int(rna_roi_mask.sum()))

# =========================================================
# (1) Per-exon summaries (mean, sum, exon_len) per track
# =========================================================
per_exon_rows = []
for exon_idx, ex in enumerate(tusc3_exons, start=1):
    if not interval.contains(ex):
        raise ValueError(f"Exon not fully inside prediction interval: {ex}")

    rel_start = ex.start - interval.start
    rel_end   = ex.end   - interval.start
    exon_len  = rel_end - rel_start
    if exon_len <= 0:
        raise ValueError(f"Exon length <= 0 for exon {exon_idx}: {ex}")

    X = Vref_rna[rel_start:rel_end, :]  # (exon_len, tracks)

    df_ex = meta_rna.copy()
    df_ex["exon_number"] = exon_idx
    df_ex["exon_chrom"]  = ex.chromosome
    df_ex["exon_start"]  = ex.start
    df_ex["exon_end"]    = ex.end
    df_ex["exon_strand"] = ex.strand
    df_ex["exon_len_bp"] = exon_len
    df_ex["exon_mean"]   = X.mean(axis=0)
    df_ex["exon_sum"]    = X.sum(axis=0)

    per_exon_rows.append(df_ex)

per_exon_rna_summary = pd.concat(per_exon_rows, ignore_index=True)

print("\nPer-exon RNA-seq summary table shape:", per_exon_rna_summary.shape)
display(per_exon_rna_summary.head(10))

# =========================================================
# (2) Gene-level summaries over UNION of exons (per track)
# =========================================================
Vref_rna_exon_union = Vref_rna[rna_roi_mask, :]  # (total_exon_bp, tracks)

gene_rna_summary = meta_rna.copy()
gene_rna_summary["gene"] = "TUSC3"
gene_rna_summary["n_exons"] = len(tusc3_exons)
gene_rna_summary["total_exon_bp"] = int(rna_roi_mask.sum())

gene_rna_summary["TUSC3_ref_exon_union_mean"] = Vref_rna_exon_union.mean(axis=0)
gene_rna_summary["TUSC3_ref_exon_union_sum"]  = Vref_rna_exon_union.sum(axis=0)
gene_rna_summary["TUSC3_ref_exon_union_p95"]  = np.quantile(Vref_rna_exon_union, 0.95, axis=0)
gene_rna_summary["TUSC3_ref_exon_union_max"]  = Vref_rna_exon_union.max(axis=0)

print("\nGene-level (union of exons) RNA-seq summary shape:", gene_rna_summary.shape)
display(gene_rna_summary.head(10))

# =========================================================
# (3) Collapsed gene-level summary across tracks
# =========================================================
cols_to_collapse = [
    "TUSC3_ref_exon_union_mean",
    "TUSC3_ref_exon_union_sum",
    "TUSC3_ref_exon_union_p95",
    "TUSC3_ref_exon_union_max",
]

collapsed_rows = []
for col in cols_to_collapse:
    v = gene_rna_summary[col].to_numpy(dtype=float)
    collapsed_rows.append({
        "gene": "TUSC3",
        "metric": col,
        "n_exons": int(gene_rna_summary["n_exons"].iloc[0]),
        "total_exon_bp": int(gene_rna_summary["total_exon_bp"].iloc[0]),
        "n_tracks": int(np.isfinite(v).sum()),
        "mean_across_tracks": float(np.nanmean(v)),
        "median_across_tracks": float(np.nanmedian(v)),
        "sd_across_tracks": float(np.nanstd(v, ddof=1)),
        "min_across_tracks": float(np.nanmin(v)),
        "max_across_tracks": float(np.nanmax(v)),
        "p05_across_tracks": float(np.nanquantile(v, 0.05)),
        "p95_across_tracks": float(np.nanquantile(v, 0.95)),
    })

gene_rna_summary_collapsed = pd.DataFrame(collapsed_rows)

print("\nCollapsed gene-level RNA-seq summary (REF):")
display(gene_rna_summary_collapsed)

RNA-seq values shape (pos, tracks): (1048576, 243)
Number of exons: 10
Total exonic bp (union): 3683

Per-exon RNA-seq summary table shape: (2430, 20)


,name,strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,gtex_tissue,data_source,endedness,genetically_modified,nonzero_mean,exon_number,exon_chrom,exon_start,exon_end,exon_strand,exon_len_bp,exon_mean,exon_sum
0,CL:0000047 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000047,neuronal stem cell,in_vitro_differentiated_cells,embryonic,,encode,paired,False,0.143617,1,chr8,15540222,15540568,+,346,2.505127,866.773804
1,CL:0000062 total RNA-seq,+,total RNA-seq,CL:0000062,osteoblast,primary_cell,adult,,encode,paired,False,0.094144,1,chr8,15540222,15540568,+,346,2.434926,842.484375
2,CL:0000138 total RNA-seq,+,total RNA-seq,CL:0000138,chondrocyte,in_vitro_differentiated_cells,embryonic,,encode,single,False,0.143112,1,chr8,15540222,15540568,+,346,1.827335,632.257812
3,CL:0000182 total RNA-seq,+,total RNA-seq,CL:0000182,hepatocyte,in_vitro_differentiated_cells,embryonic,,encode,paired,False,0.117008,1,chr8,15540222,15540568,+,346,3.016108,1043.573486
4,CL:0000307 total RNA-seq,+,total RNA-seq,CL:0000307,tracheal epithelial cell,primary_cell,adult,,encode,paired,False,0.243777,1,chr8,15540222,15540568,+,346,4.191716,1450.333740
5,CL:0000312 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000312,keratinocyte,primary_cell,unknown,,encode,paired,False,0.158347,1,chr8,15540222,15540568,+,346,1.190371,411.868286
6,CL:0000515 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000515,skeletal muscle myoblast,primary_cell,unknown,,encode,paired,False,0.236331,1,chr8,15540222,15540568,+,346,1.700074,588.225769
7,CL:0000515 total RNA-seq,+,total RNA-seq,CL:0000515,skeletal muscle myoblast,primary_cell,unknown,,encode,paired,False,0.214407,1,chr8,15540222,15540568,+,346,2.710504,937.834290
8,CL:0000594 total RNA-seq,+,total RNA-seq,CL:0000594,skeletal muscle satellite cell,primary_cell,adult,,encode,paired,False,0.151521,1,chr8,15540222,15540568,+,346,2.639936,913.417969
9,CL:0000623 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000623,natural killer cell,primary_cell,adult,,encode,paired,False,0.098101,1,chr8,15540222,15540568,+,346,0.004566,1.579799



Gene-level (union of exons) RNA-seq summary shape: (243, 19)


,name,strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,gtex_tissue,data_source,endedness,genetically_modified,nonzero_mean,gene,n_exons,total_exon_bp,TUSC3_ref_exon_union_mean,TUSC3_ref_exon_union_sum,TUSC3_ref_exon_union_p95,TUSC3_ref_exon_union_max
0,CL:0000047 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000047,neuronal stem cell,in_vitro_differentiated_cells,embryonic,,encode,paired,False,0.143617,TUSC3,10,3683,2.253224,8298.625000,5.875000,6.718750
1,CL:0000062 total RNA-seq,+,total RNA-seq,CL:0000062,osteoblast,primary_cell,adult,,encode,paired,False,0.094144,TUSC3,10,3683,0.938812,3457.643555,2.890625,3.953125
2,CL:0000138 total RNA-seq,+,total RNA-seq,CL:0000138,chondrocyte,in_vitro_differentiated_cells,embryonic,,encode,single,False,0.143112,TUSC3,10,3683,1.344769,4952.786133,4.156250,4.781250
3,CL:0000182 total RNA-seq,+,total RNA-seq,CL:0000182,hepatocyte,in_vitro_differentiated_cells,embryonic,,encode,paired,False,0.117008,TUSC3,10,3683,1.705789,6282.420898,4.593750,5.312500
4,CL:0000307 total RNA-seq,+,total RNA-seq,CL:0000307,tracheal epithelial cell,primary_cell,adult,,encode,paired,False,0.243777,TUSC3,10,3683,1.356929,4997.569336,4.625000,6.937500
5,CL:0000312 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000312,keratinocyte,primary_cell,unknown,,encode,paired,False,0.158347,TUSC3,10,3683,1.521826,5604.885742,5.468750,6.875000
6,CL:0000515 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000515,skeletal muscle myoblast,primary_cell,unknown,,encode,paired,False,0.236331,TUSC3,10,3683,1.557640,5736.787598,5.281250,6.625000
7,CL:0000515 total RNA-seq,+,total RNA-seq,CL:0000515,skeletal muscle myoblast,primary_cell,unknown,,encode,paired,False,0.214407,TUSC3,10,3683,1.375309,5065.263672,3.812500,4.593750
8,CL:0000594 total RNA-seq,+,total RNA-seq,CL:0000594,skeletal muscle satellite cell,primary_cell,adult,,encode,paired,False,0.151521,TUSC3,10,3683,0.901190,3319.082031,2.796875,4.500000
9,CL:0000623 polyA plus RNA-seq,+,polyA plus RNA-seq,CL:0000623,natural killer cell,primary_cell,adult,,encode,paired,False,0.098101,TUSC3,10,3683,0.005953,21.923414,0.020752,0.076172



Collapsed gene-level RNA-seq summary (REF):


,gene,metric,n_exons,total_exon_bp,n_tracks,mean_across_tracks,median_across_tracks,sd_across_tracks,min_across_tracks,max_across_tracks,p05_across_tracks,p95_across_tracks
0,TUSC3,TUSC3_ref_exon_union_mean,10,3683,243,0.570121,0.317575,0.688690,0.000015,2.891696,0.000165,1.836722
1,TUSC3,TUSC3_ref_exon_union_sum,10,3683,243,2099.755025,1169.629517,2536.445056,0.054535,10650.117188,0.608935,6764.648193
2,TUSC3,TUSC3_ref_exon_union_p95,10,3683,243,1.760530,1.109375,2.062624,0.000072,7.843750,0.000816,5.831250
3,TUSC3,TUSC3_ref_exon_union_max,10,3683,243,2.254912,1.445312,2.593995,0.000114,10.250000,0.003658,7.284375


In [21]:
############################################################
# Save RNA-seq coverage summaries to TXT files
############################################################

import os

OUTDIR = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/RNA_seq_coverage"

# create directory if it doesn't exist
os.makedirs(OUTDIR, exist_ok=True)

# define output paths
per_exon_path = os.path.join(OUTDIR, "TUSC3_per_exon_rna_seq_REF.txt")
gene_path     = os.path.join(OUTDIR, "TUSC3_gene_level_rna_seq_REF.txt")
collapsed_path = os.path.join(OUTDIR, "TUSC3_gene_level_collapsed_rna_seq_REF.txt")

# save as tab-delimited txt
per_exon_rna_summary.to_csv(per_exon_path, sep="\t", index=False)
gene_rna_summary.to_csv(gene_path, sep="\t", index=False)
gene_rna_summary_collapsed.to_csv(collapsed_path, sep="\t", index=False)

print("Saved files:")
print(per_exon_path)
print(gene_path)
print(collapsed_path)

Saved files:
/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/RNA_seq_coverage/TUSC3_per_exon_rna_seq_REF.txt
/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/RNA_seq_coverage/TUSC3_gene_level_rna_seq_REF.txt
/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/RNA_seq_coverage/TUSC3_gene_level_collapsed_rna_seq_REF.txt


In [22]:
############################################################
# Step 5B (REF): Summarize CAGE signal in the 501-bp TSS ROI
#   Outputs:
#     1) cage_roi_summary            (per-track over 501bp ROI)
#     2) cage_roi_summary_collapsed  (collapsed across tracks)
############################################################

import numpy as np
import pandas as pd

# --- inputs assumed ---
# out_ref   : AlphaGenome Output (reference prediction)
# interval  : 1Mb genome.Interval
# cage_roi  : genome.Interval (501bp around TUSC3 TSS)

ref_cage_t = out_ref.cage
Vref_cage = np.asarray(ref_cage_t.values)  # (pos, tracks)
meta_cage = ref_cage_t.metadata.copy().reset_index(drop=True)

# --- sanity checks ---
assert Vref_cage.shape[0] == interval.width, (Vref_cage.shape[0], interval.width)

# relative coordinates within 1Mb interval
rel_start = cage_roi.start - interval.start
rel_end   = cage_roi.end   - interval.start

if not (0 <= rel_start < rel_end <= interval.width):
    raise ValueError(
        f"cage_roi must be fully contained within interval.\n"
        f"cage_roi={cage_roi}\ninterval={interval}\n"
        f"relative={rel_start}-{rel_end}"
    )

assert (rel_end - rel_start) == 501, (rel_end - rel_start)

print("CAGE values shape (pos, tracks):", Vref_cage.shape)
print("CAGE ROI (relative):", f"{rel_start}-{rel_end}", "| width:", rel_end - rel_start)

# =========================================================
# (1) Per-track summaries over the 501bp ROI
# =========================================================
X = Vref_cage[rel_start:rel_end, :]  # (501, tracks)

cage_roi_summary = meta_cage.copy()
cage_roi_summary["gene"] = "TUSC3"
cage_roi_summary["roi_name"] = "TUSC3_TSS_501bp"
cage_roi_summary["roi_bp"] = int(rel_end - rel_start)

cage_roi_summary["TUSC3_ref_CAGE_mean_501bp"] = X.mean(axis=0)
cage_roi_summary["TUSC3_ref_CAGE_sum_501bp"]  = X.sum(axis=0)
cage_roi_summary["TUSC3_ref_CAGE_p95_501bp"]  = np.quantile(X, 0.95, axis=0)
cage_roi_summary["TUSC3_ref_CAGE_max_501bp"]  = X.max(axis=0)

print("\nCAGE ROI per-track summary shape:", cage_roi_summary.shape)
display(cage_roi_summary.head(10))

# =========================================================
# (2) Collapsed summary across tracks
# =========================================================
cols_to_collapse = [
    "TUSC3_ref_CAGE_mean_501bp",
    "TUSC3_ref_CAGE_sum_501bp",
    "TUSC3_ref_CAGE_p95_501bp",
    "TUSC3_ref_CAGE_max_501bp",
]

collapsed_rows = []
for col in cols_to_collapse:
    v = cage_roi_summary[col].to_numpy(dtype=float)
    collapsed_rows.append({
        "gene": "TUSC3",
        "roi_name": "TUSC3_TSS_501bp",
        "roi_bp": int(rel_end - rel_start),
        "metric": col,
        "n_tracks": int(np.isfinite(v).sum()),
        "mean_across_tracks": float(np.nanmean(v)),
        "median_across_tracks": float(np.nanmedian(v)),
        "sd_across_tracks": float(np.nanstd(v, ddof=1)),
        "min_across_tracks": float(np.nanmin(v)),
        "max_across_tracks": float(np.nanmax(v)),
        "p05_across_tracks": float(np.nanquantile(v, 0.05)),
        "p95_across_tracks": float(np.nanquantile(v, 0.95)),
    })

cage_roi_summary_collapsed = pd.DataFrame(collapsed_rows)

print("\nCollapsed CAGE ROI summary (REF):")
display(cage_roi_summary_collapsed)

CAGE values shape (pos, tracks): (1048576, 166)
CAGE ROI (relative): 355504-356005 | width: 501

CAGE ROI per-track summary shape: (166, 15)


,name,strand,Assay title,ontology_curie,biosample_name,biosample_type,data_source,nonzero_mean,gene,roi_name,roi_bp,TUSC3_ref_CAGE_mean_501bp,TUSC3_ref_CAGE_sum_501bp,TUSC3_ref_CAGE_p95_501bp,TUSC3_ref_CAGE_max_501bp
0,LQhCAGE CL:0000895,+,LQhCAGE,CL:0000895,"naive thymus-derived CD4-positive, alpha-beta ...",primary_cell,fantom,91.958466,TUSC3,TUSC3_TSS_501bp,501,0.483701,242.334198,0.285156,203.0
1,hCAGE CL:0000047,+,hCAGE,CL:0000047,neuronal stem cell,primary_cell,fantom,12.134685,TUSC3,TUSC3_TSS_501bp,501,5.368959,2689.848633,28.625000,249.0
2,hCAGE CL:0000062,+,hCAGE,CL:0000062,osteoblast,primary_cell,fantom,30.618631,TUSC3,TUSC3_TSS_501bp,501,13.884708,6956.238770,67.000000,968.0
3,hCAGE CL:0000138,+,hCAGE,CL:0000138,chondrocyte,primary_cell,fantom,21.439882,TUSC3,TUSC3_TSS_501bp,501,11.076879,5549.516113,52.750000,840.0
4,hCAGE CL:0000182,+,hCAGE,CL:0000182,hepatocyte,primary_cell,fantom,34.048721,TUSC3,TUSC3_TSS_501bp,501,5.376755,2693.754150,19.750000,556.0
5,hCAGE CL:0000307,+,hCAGE,CL:0000307,tracheal epithelial cell,primary_cell,fantom,43.871811,TUSC3,TUSC3_TSS_501bp,501,10.384685,5202.727051,46.750000,984.0
6,hCAGE CL:0000312,+,hCAGE,CL:0000312,keratinocyte,primary_cell,fantom,33.141487,TUSC3,TUSC3_TSS_501bp,501,11.093424,5557.805176,54.500000,960.0
7,hCAGE CL:0000515,+,hCAGE,CL:0000515,skeletal muscle myoblast,primary_cell,fantom,33.867573,TUSC3,TUSC3_TSS_501bp,501,14.437332,7233.103516,71.500000,1008.0
8,hCAGE CL:0000594,+,hCAGE,CL:0000594,skeletal muscle satellite cell,primary_cell,fantom,25.070316,TUSC3,TUSC3_TSS_501bp,501,15.690502,7860.941406,86.500000,1024.0
9,hCAGE CL:0000623,+,hCAGE,CL:0000623,natural killer cell,primary_cell,fantom,17.459036,TUSC3,TUSC3_TSS_501bp,501,0.310949,155.785599,0.154297,129.0



Collapsed CAGE ROI summary (REF):


,gene,roi_name,roi_bp,metric,n_tracks,mean_across_tracks,median_across_tracks,sd_across_tracks,min_across_tracks,max_across_tracks,p05_across_tracks,p95_across_tracks
0,TUSC3,TUSC3_TSS_501bp,501,TUSC3_ref_CAGE_mean_501bp,166,6.022605,0.200608,7.216954,0.000348,30.318905,0.008015,18.976973
1,TUSC3,TUSC3_TSS_501bp,501,TUSC3_ref_CAGE_sum_501bp,166,3017.325122,100.504635,3615.694228,0.174254,15189.771484,4.015431,9507.463867
2,TUSC3,TUSC3_TSS_501bp,501,TUSC3_ref_CAGE_p95_501bp,166,28.134658,0.264648,35.264030,0.001030,150.000000,0.026245,89.875000
3,TUSC3,TUSC3_TSS_501bp,501,TUSC3_ref_CAGE_max_501bp,166,484.169804,66.101562,547.650916,0.014221,2688.000000,0.307129,1302.000000


In [23]:
############################################################
# Save CAGE coverage summaries to TXT files
############################################################

import os

OUTDIR_CAGE = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/CAGE_coverage"
os.makedirs(OUTDIR_CAGE, exist_ok=True)

cage_track_path = os.path.join(OUTDIR_CAGE, "TUSC3_CAGE_TSS501bp_REF.txt")
cage_collapsed_path = os.path.join(OUTDIR_CAGE, "TUSC3_CAGE_TSS501bp_collapsed_REF.txt")

# Save as tab-delimited TXT
cage_roi_summary.to_csv(cage_track_path, sep="\t", index=False)
cage_roi_summary_collapsed.to_csv(cage_collapsed_path, sep="\t", index=False)

print("Saved files:")
print(cage_track_path)
print(cage_collapsed_path)

Saved files:
/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/CAGE_coverage/TUSC3_CAGE_TSS501bp_REF.txt
/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/CAGE_coverage/TUSC3_CAGE_TSS501bp_collapsed_REF.txt


In [33]:
############################################################
# Step 6 (loop, memory-friendly): Process RANDOM_01..RANDOM_10
#   - Load 1 random .pkl at a time
#   - Summarize RNA-seq (PER-EXON + gene-level) + CAGE (TSS 501bp)
#   - Save TXT outputs
#   - Delete big objects each iteration to free memory
############################################################

import os
import gc
import pickle
import numpy as np
import pandas as pd

# --- inputs assumed already defined ---
# interval      : 1Mb genome.Interval
# tusc3_exons   : list[genome.Interval]
# rna_roi_mask  : np.ndarray[bool] length == interval.width
# cage_roi      : genome.Interval (501bp around TSS)

PRED_DIR = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/random10"
OUTDIR_RNA = os.path.join(PRED_DIR, "RNA_seq_coverage")
OUTDIR_CAGE = os.path.join(PRED_DIR, "CAGE_coverage")
os.makedirs(OUTDIR_RNA, exist_ok=True)
os.makedirs(OUTDIR_CAGE, exist_ok=True)

# constant relative coords for CAGE ROI within 1Mb interval
cage_rel_start = cage_roi.start - interval.start
cage_rel_end   = cage_roi.end   - interval.start
assert (cage_rel_end - cage_rel_start) == 501
assert 0 <= cage_rel_start < cage_rel_end <= interval.width

def summarize_one_output(out_obj, tag: str):
    """Summarize RNA-seq (per-exon + union) and CAGE (TSS501bp) for one Output object."""

    # ------------------ RNA-seq ------------------
    rna_t = out_obj.rna_seq
    Vrna = np.asarray(rna_t.values)  # (pos, tracks)
    meta_rna = rna_t.metadata.copy().reset_index(drop=True)

    assert Vrna.shape[0] == interval.width

    # (A) per-exon (long format)
    per_exon_rows = []
    for exon_idx, ex in enumerate(tusc3_exons, start=1):
        if not interval.contains(ex):
            raise ValueError(f"Exon not fully inside prediction interval: {ex}")

        rel_start = ex.start - interval.start
        rel_end   = ex.end   - interval.start
        exon_len  = rel_end - rel_start
        if exon_len <= 0:
            raise ValueError(f"Exon length <= 0 for exon {exon_idx}: {ex}")

        X = Vrna[rel_start:rel_end, :]  # (exon_len, tracks)

        df_ex = meta_rna.copy()
        df_ex["gene"] = "TUSC3"
        df_ex["tag"] = tag
        df_ex["exon_number"] = exon_idx
        df_ex["exon_chrom"]  = ex.chromosome
        df_ex["exon_start"]  = ex.start
        df_ex["exon_end"]    = ex.end
        df_ex["exon_strand"] = ex.strand
        df_ex["exon_len_bp"] = exon_len

        df_ex[f"TUSC3_{tag}_exon_mean"] = X.mean(axis=0)
        df_ex[f"TUSC3_{tag}_exon_sum"]  = X.sum(axis=0)

        per_exon_rows.append(df_ex)

    per_exon_rna = pd.concat(per_exon_rows, ignore_index=True)

    # (B) gene-level over UNION of exons (per track)
    assert rna_roi_mask.shape[0] == interval.width and rna_roi_mask.sum() > 0
    Vrna_ex_union = Vrna[rna_roi_mask, :]  # (exon_bp, tracks)

    gene_rna = meta_rna.copy()
    gene_rna["gene"] = "TUSC3"
    gene_rna["tag"] = tag
    gene_rna["n_exons"] = len(tusc3_exons)
    gene_rna["total_exon_bp"] = int(rna_roi_mask.sum())

    gene_rna[f"TUSC3_{tag}_exon_union_mean"] = Vrna_ex_union.mean(axis=0)
    gene_rna[f"TUSC3_{tag}_exon_union_sum"]  = Vrna_ex_union.sum(axis=0)
    gene_rna[f"TUSC3_{tag}_exon_union_p95"]  = np.quantile(Vrna_ex_union, 0.95, axis=0)
    gene_rna[f"TUSC3_{tag}_exon_union_max"]  = Vrna_ex_union.max(axis=0)

    # (C) collapsed across tracks
    cols_rna = [
        f"TUSC3_{tag}_exon_union_mean",
        f"TUSC3_{tag}_exon_union_sum",
        f"TUSC3_{tag}_exon_union_p95",
        f"TUSC3_{tag}_exon_union_max",
    ]
    rna_collapsed = []
    for col in cols_rna:
        v = gene_rna[col].to_numpy(dtype=float)
        rna_collapsed.append({
            "gene": "TUSC3",
            "tag": tag,
            "metric": col,
            "n_exons": int(gene_rna["n_exons"].iloc[0]),
            "total_exon_bp": int(gene_rna["total_exon_bp"].iloc[0]),
            "n_tracks": int(np.isfinite(v).sum()),
            "mean_across_tracks": float(np.nanmean(v)),
            "median_across_tracks": float(np.nanmedian(v)),
            "sd_across_tracks": float(np.nanstd(v, ddof=1)),
            "min_across_tracks": float(np.nanmin(v)),
            "max_across_tracks": float(np.nanmax(v)),
            "p05_across_tracks": float(np.nanquantile(v, 0.05)),
            "p95_across_tracks": float(np.nanquantile(v, 0.95)),
        })
    gene_rna_collapsed = pd.DataFrame(rna_collapsed)

    # ------------------ CAGE ------------------
    cage_t = out_obj.cage
    Vcage = np.asarray(cage_t.values)  # (pos, tracks)
    meta_cage = cage_t.metadata.copy().reset_index(drop=True)

    assert Vcage.shape[0] == interval.width
    Xc = Vcage[cage_rel_start:cage_rel_end, :]  # (501, tracks)

    cage_sum = meta_cage.copy()
    cage_sum["gene"] = "TUSC3"
    cage_sum["tag"] = tag
    cage_sum["roi_name"] = "TUSC3_TSS_501bp"
    cage_sum["roi_bp"] = 501

    cage_sum[f"TUSC3_{tag}_CAGE_mean_501bp"] = Xc.mean(axis=0)
    cage_sum[f"TUSC3_{tag}_CAGE_sum_501bp"]  = Xc.sum(axis=0)
    cage_sum[f"TUSC3_{tag}_CAGE_p95_501bp"]  = np.quantile(Xc, 0.95, axis=0)
    cage_sum[f"TUSC3_{tag}_CAGE_max_501bp"]  = Xc.max(axis=0)

    cols_cage = [
        f"TUSC3_{tag}_CAGE_mean_501bp",
        f"TUSC3_{tag}_CAGE_sum_501bp",
        f"TUSC3_{tag}_CAGE_p95_501bp",
        f"TUSC3_{tag}_CAGE_max_501bp",
    ]
    cage_collapsed = []
    for col in cols_cage:
        v = cage_sum[col].to_numpy(dtype=float)
        cage_collapsed.append({
            "gene": "TUSC3",
            "tag": tag,
            "roi_name": "TUSC3_TSS_501bp",
            "roi_bp": 501,
            "metric": col,
            "n_tracks": int(np.isfinite(v).sum()),
            "mean_across_tracks": float(np.nanmean(v)),
            "median_across_tracks": float(np.nanmedian(v)),
            "sd_across_tracks": float(np.nanstd(v, ddof=1)),
            "min_across_tracks": float(np.nanmin(v)),
            "max_across_tracks": float(np.nanmax(v)),
            "p05_across_tracks": float(np.nanquantile(v, 0.05)),
            "p95_across_tracks": float(np.nanquantile(v, 0.95)),
        })
    cage_collapsed_df = pd.DataFrame(cage_collapsed)

    return per_exon_rna, gene_rna, gene_rna_collapsed, cage_sum, cage_collapsed_df


# -------- main loop --------
for i in range(1, 11):  # RANDOM_01 .. RANDOM_10
    tag = f"rand{i:02d}"
    pkl_path = os.path.join(PRED_DIR, f"alphagenome_out_rand_{i:02d}.pkl")

    if not os.path.exists(pkl_path):
        print(f"[skip] missing: {pkl_path}")
        continue

    print(f"\n[{i:02d}/10] Loading {os.path.basename(pkl_path)} ...")
    with open(pkl_path, "rb") as f:
        out_i = pickle.load(f)

    print(f"[{i:02d}/10] Summarizing {tag} (RNA per-exon + gene, CAGE ROI) ...")
    per_exon_rna_i, gene_rna_i, gene_rna_collapsed_i, cage_i, cage_collapsed_i = summarize_one_output(out_i, tag)

    # --- save RNA-seq ---
    per_exon_rna_i.to_csv(
        os.path.join(OUTDIR_RNA, f"TUSC3_per_exon_rna_seq_RANDOM_{i:02d}.txt"),
        sep="\t", index=False
    )
    gene_rna_i.to_csv(
        os.path.join(OUTDIR_RNA, f"TUSC3_gene_level_rna_seq_RANDOM_{i:02d}.txt"),
        sep="\t", index=False
    )
    gene_rna_collapsed_i.to_csv(
        os.path.join(OUTDIR_RNA, f"TUSC3_gene_level_collapsed_rna_seq_RANDOM_{i:02d}.txt"),
        sep="\t", index=False
    )

    # --- save CAGE ---
    cage_i.to_csv(
        os.path.join(OUTDIR_CAGE, f"TUSC3_CAGE_TSS501bp_RANDOM_{i:02d}.txt"),
        sep="\t", index=False
    )
    cage_collapsed_i.to_csv(
        os.path.join(OUTDIR_CAGE, f"TUSC3_CAGE_TSS501bp_collapsed_RANDOM_{i:02d}.txt"),
        sep="\t", index=False
    )

    print(f"[{i:02d}/10] Saved summaries.")

    # --- free memory aggressively ---
    del out_i, per_exon_rna_i, gene_rna_i, gene_rna_collapsed_i, cage_i, cage_collapsed_i
    gc.collect()

print("\nDone. All available RANDOM outputs processed (with per-exon RNA-seq included).")




[01/10] Loading alphagenome_out_rand_01.pkl ...
[01/10] Summarizing rand01 (RNA per-exon + gene, CAGE ROI) ...
[01/10] Saved summaries.

[02/10] Loading alphagenome_out_rand_02.pkl ...
[02/10] Summarizing rand02 (RNA per-exon + gene, CAGE ROI) ...
[02/10] Saved summaries.

[03/10] Loading alphagenome_out_rand_03.pkl ...
[03/10] Summarizing rand03 (RNA per-exon + gene, CAGE ROI) ...
[03/10] Saved summaries.

[04/10] Loading alphagenome_out_rand_04.pkl ...
[04/10] Summarizing rand04 (RNA per-exon + gene, CAGE ROI) ...
[04/10] Saved summaries.

[05/10] Loading alphagenome_out_rand_05.pkl ...
[05/10] Summarizing rand05 (RNA per-exon + gene, CAGE ROI) ...
[05/10] Saved summaries.

[06/10] Loading alphagenome_out_rand_06.pkl ...
[06/10] Summarizing rand06 (RNA per-exon + gene, CAGE ROI) ...
[06/10] Saved summaries.

[07/10] Loading alphagenome_out_rand_07.pkl ...
[07/10] Summarizing rand07 (RNA per-exon + gene, CAGE ROI) ...
[07/10] Saved summaries.

[08/10] Loading alphagenome_out_rand_08.